# Anormaly detection with IsolationForrest

In [1]:
!pip install pandas scikit-learn

import pandas as pd
from sklearn.ensemble import IsolationForest


Branddata = pd.read_csv("Branddata.csv")


categorical_vars = [
    "ROOF_TYPE",
    "WATER_SUPPLY_TYPE",
    "HEATING_TYPE",
    "OUTER_WALLS"
]

Branddata_encoded = pd.get_dummies(
    Branddata,
    columns=categorical_vars,
    dummy_na=False
)


model_data = Branddata_encoded.select_dtypes(include=["number"])


anomaly_results = pd.DataFrame(index=Branddata.index)


for var in model_data.columns:

    print(f"Running Isolation Forest for: {var}")

    # Ignore NA values ONLY for this variable
    X = model_data[[var]].dropna()

    iso = IsolationForest(
        n_estimators=100,
        contamination=0.01,
        random_state=42
    )

    predictions = iso.fit_predict(X)

    scores = iso.decision_function(X)

    anomaly_results[f"{var}_anomaly"] = pd.NA
    anomaly_results[f"{var}_score"] = pd.NA

    anomaly_results.loc[
        X.index,
        f"{var}_anomaly"
    ] = predictions

    anomaly_results.loc[
        X.index,
        f"{var}_score"
    ] = scores

Running Isolation Forest for: POLICY
Running Isolation Forest for: EXPOSURE


KeyboardInterrupt: 

In [ ]:
var = "RESIDENTIAL_AREA"

result = pd.concat(
    [
        Branddata[[var]],
        anomaly_results[
            [f"{var}_score"]
        ]
    ],
    axis=1
)

result = result.dropna()

result = result.sort_values(
    f"{var}_score"
)

# Show most anomalous observations
print(result.head(20))